### Bronze Olist Metadata Update

In [ ]:
# This notebook is used in the bronze layer orchestration pipeline to
# update the metadata tables to help facilitate incremental loading
# patterns and failure alerting

# Developer:    Asif Shah
# Created Date: 14.09.2025

In [ ]:
# Update the last_rows_check and audit_control tables to support incremental loading patterns

# 1. Fetch all unique Folder and Table combinations waiting for row counts
table_rows_req_df = spark.sql("""
    SELECT DISTINCT folder_name, table_name 
    FROM dbo.audit_control 
    WHERE table_rows = 0
""")

# 1. Clear the target table
spark.sql("TRUNCATE TABLE dbo.last_rows_check")

# 2. Insert the aggregated data
spark.sql("""
    INSERT INTO dbo.last_rows_check
    SELECT folder_name, SUM(file_rows), 0, 0
    FROM dbo.audit_control 
    WHERE table_rows = 0
    GROUP BY folder_name
""")


rows = table_rows_req_df.collect()

# 3. Iterate through the pending records
for row in rows:
    folder = row["folder_name"]
    tablename = row["table_name"]
    
    try:
        # Fetch actual row count
        count_df = spark.sql(f"SELECT COUNT(*) as total_rows FROM Bronze.{tablename}")
        actual_count = count_df.first()["total_rows"]
        
        # Statement 1: Update audit control (Added single quotes around string variables)
        spark.sql(f"""
            UPDATE dbo.audit_control 
            SET table_rows = {actual_count}, 
                status = 'Completed', 
                processed_timestamp = current_timestamp() 
            WHERE folder_name = '{folder}' 
              AND table_name = '{tablename}' 
              AND table_rows = 0
        """)

    except Exception as e:
        print(f"There were errors updating the audit tables for {folder}.{tablename}. Error: {str(e)}")

     
    

In [ ]:
# Updating the dbo.last_rows_check table to get the total number of rows in the 
# the latest bronze tables data loads to use as a check in the silver orchestration
# pipeline

# 1. Update table_rows by joining and aggregating audit_control
spark.sql("""
    MERGE INTO dbo.last_rows_check AS target
    USING (
        SELECT folder_name, SUM(table_rows) AS total_table_rows
        FROM dbo.audit_control
        GROUP BY folder_name
    ) AS source
    ON target.folder_name = source.folder_name
    WHEN MATCHED THEN
        UPDATE SET target.table_rows = source.total_table_rows
""")

# 2. Update rows_diff using the freshly updated values
spark.sql("""
    UPDATE dbo.last_rows_check
    SET row_diff = file_rows - table_rows
""")

